## Face Recognition with InsightFace
This notebook demonstrates how to implement face detection and recognition using InsightFace. We utilize pre-trained models for detection and recognition, extract embeddings from a given dataset, and perform real-time face recognition. 
Since Kaggle does not support webcam access, you can adapt the code to use images or video files instead.

### Dataset Structure
The dataset root directory is named Dataset, and inside it, each subdirectory corresponds to a person's name. Each subdirectory contains images of that person. The structure looks like this:

```
Dataset/
├── Person1/
│   ├── image1.jpg
│   ├── image2.jpg
│   ├── image3.jpg
│   └── ...
├── Person2/
│   ├── image1.jpg
│   ├── image2.jpg
│   ├── ...
├── Person3/
│   ├── image1.jpg
│   ├── ...
└── ...
```

### Pretrained Models

The detection and recognition models used in this project can be downloaded from the official InsightFace GitHub repository:

➡️ [InsightFace Models - GitHub](https://github.com/deepinsight/insightface/tree/master/python-package)


!pip install insightface onnx onnxruntime-gpu

In [1]:
import os # Gestion des fichiers systèmes
import cv2 # Capture webcam
import pickle # Gestion des sauvegardes de poids
import numpy as np # Stockage des embeddings et calcul des similarités cosinus
import json
from glob import glob # Recherche par motif
from insightface.app.common import Face # Construction de visage
from insightface.model_zoo import model_zoo # Chargement des modèles
from datetime import datetime

In [2]:
# detection and recognition models
#det_model_path = '/kaggle/input/insightface-buffalo_l/onnx/default/1/det_10g.onnx'  # RetinaFace-10GF for detection
#rec_model_path = '/kaggle/input/insightface-buffalo_l/onnx/default/1/w600k_r50.onnx'  # ResNet50@WebFace600K for recognition
# Chargement et téléchargement des modèles s'ils ne sont pas détectés
det_model = model_zoo.get_model('buffalo_l/det_10g.onnx', download=True)
rec_model = model_zoo.get_model('buffalo_l/w600k_r50.onnx', download=True)

# Initialisation du modèle de détection
det_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)
rec_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


In [3]:
def extract_face_embeddings():
    embeddings_path = 'known_embeddings.npy'  # embeddings path
    names_path = 'known_names.pkl'  # names path

    # Chargement ou initialisation de la base de personnes
    if os.path.exists(embeddings_path) and os.path.exists(names_path):
        known_embeddings = np.load(embeddings_path)
        with open(names_path, 'rb') as f:
            known_names = pickle.load(f)
    else:
        known_embeddings = np.array([]).reshape(0, 512)  # empty array for embeddings
        known_names = []  # empty list for names
    
    # Direction vers le dataset
    person_dirs = [d for d in os.listdir('Dataset') if os.path.isdir(os.path.join('Dataset', d))]
    print("Extracting embeddings...")

    for person_name in person_dirs:
        directory = os.path.join('Dataset', person_name)
        img_paths = glob(f'{directory}/*.jpg')
        new_embeddings = []  # liste des embeddings par personne

        for img_path in img_paths:
            img = cv2.imread(img_path)
            if img is None:
                print(f"  Impossible de lire {img_path}")
                continue

            # On détecte le visage, et on renvoie lespoints clés (yeux, coins de bouche, nez, landmarks)
            bboxes, kpss = det_model.detect(img, max_num=0, metric='default')
            if len(bboxes) == 0:
                print(f"  Aucun visage détecté dans {img_path}")
                continue
            
            # Suppose que le premier visage détecté est correct
            bbox = bboxes[0, :4]
            det_score = bboxes[0, 4]
            kps = kpss[0]
            face = Face(bbox=bbox, kps=kps, det_score=det_score)
            rec_model.get(img, face)  # on calcule l'embedding du visage
            
            if hasattr(face, 'normed_embedding'):
                new_embeddings.append(face.normed_embedding)  # Stockage de l'embedding
            else:
                print(f"Failed to extract embedding for {img_path}")
        
        # Si un nouvel embedding est trouvé pour une personne, i=on l'ajoute à la liste
        if new_embeddings:
            new_embeddings = np.vstack(new_embeddings)
            known_embeddings = np.vstack([known_embeddings, new_embeddings])
            known_names.extend([person_name] * new_embeddings.shape[0])
    
    # On sauvegarde les noms et embeddings
    np.save(embeddings_path, known_embeddings)
    with open(names_path, 'wb') as f:
        pickle.dump(known_names, f)

    # Print all embeddings at the end of the process
    print("All embeddings extracted and saved successfully.")
    print(known_embeddings)
    
extract_face_embeddings()


Extracting embeddings...


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


  Aucun visage détecté dans Dataset\NGONO\IMG_20240929_162015_598.jpg
All embeddings extracted and saved successfully.
[[-0.06973715 -0.03798878  0.0595185  ... -0.07360877 -0.03653792
  -0.04464968]
 [-0.02307108  0.00412494  0.00095518 ... -0.07462476 -0.02322724
   0.01040357]
 [-0.09043147 -0.04598134 -0.00164792 ... -0.02709156 -0.03254745
   0.05335622]
 ...
 [-0.02069126 -0.07119238 -0.03483915 ...  0.03795842 -0.00190317
  -0.02276914]
 [ 0.00634547 -0.07262027 -0.00914921 ...  0.03645123 -0.01378907
   0.01120723]
 [-0.02585141 -0.07169753 -0.02152805 ...  0.03579126 -0.00907747
   0.01121777]]


In [4]:
def load_embeddings():
    # Chargeent des embeddings
    embeddings_path = 'known_embeddings.npy'
    names_path = 'known_names.pkl'

    if os.path.exists(embeddings_path) and os.path.exists(names_path):
        known_embeddings = np.load(embeddings_path)
        with open(names_path, 'rb') as f:
            known_names = pickle.load(f)
        return known_embeddings, known_names
    else:
        print("No saved embeddings found.")
        return None, None

In [5]:
def find_match(embedding, known_embeddings, known_names, threshold=0.5):
    scores = np.dot(embedding, known_embeddings.T) # similarité cosinus entre les embeddings
    scores = np.clip(scores, 0., 1.) # On élimine les valeurs négatives et aberrantes
    idx = np.argmax(scores) #  index de score le plus élevé
    score = scores[idx]
    # Si le score excède 50%, on affiche le nom. Sinon, Inconnu
    name = known_names[idx] if score > threshold else 'Inconnu' 
    return name, score

For real-time recognition, you can use the following function. Note that this will not work directly on Kaggle since it does not support webcam access. You can modify it to use images or video files instead.

In [6]:
FICHIER_LOG = "acces.json"

def charger_logs():
    """Charge l'historique depuis acces.json. Retourne [] si absent ou corrompu."""
    if not os.path.exists(FICHIER_LOG):
        return []
    with open(FICHIER_LOG, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []

def enregistrer_acces(nom, statut, score, image=None):
    """Ajoute une entrée dans le journal.
    Paramètres : nom (str), statut ('succes'|'refusé'), confiance (float 0-1),
                 image (str|None) — nom du fichier image pour les inconnus."""
    logs = charger_logs()
    logs.append({
        "date": datetime.now().strftime("%Y-%m-%d"),
        "heure": datetime.now().strftime("%H:%M:%S"),
        "nom": nom,
        "statut": statut,
        "score_détection": round(float(score), 4),
        "image": image   # None pour les succès, nom de fichier pour les inconnus
    })
    with open(FICHIER_LOG, "w", encoding="utf-8") as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)

In [7]:
DOSSIER_INCONNUS = "inconnus"
DOSSIER_SUCCESS   = "succes"
os.makedirs(DOSSIER_INCONNUS, exist_ok=True)
os.makedirs(DOSSIER_SUCCESS,   exist_ok=True)

In [8]:
def agrandir_bbox_epaules(bbox, frame_shape, marge_haut=0.5, marge_bas=0.8, marge_cotes=0.6):
    """
    Agrandit la bounding box du visage pour inclure les épaules.
    Utilisée uniquement pour la sauvegarde de l'image, jamais pour l'embedding.
    """
    x1, y1, x2, y2 = bbox.astype(int)
    h_frame, w_frame = frame_shape[:2]

    largeur = x2 - x1
    hauteur = y2 - y1

    x1_e = max(0, x1 - int(largeur * marge_cotes))
    y1_e = max(0, y1 - int(hauteur * marge_haut))
    x2_e = min(w_frame, x2 + int(largeur * marge_cotes))
    y2_e = min(h_frame, y2 + int(hauteur * marge_bas))

    return x1_e, y1_e, x2_e, y2_e

In [9]:
def face_recognition(known_embeddings, known_names, threshold=0.5):
    cap = cv2.VideoCapture(0, cv2.CAP_MSMF) # for webcam access
    # to run on a video file:
    # cap = cv2.VideoCapture("path_to_video.mp4")
    if not cap.isOpened():
        print("Unable to open camera.")
        return
    
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("Erreur : frame non capturée")
            break

        bboxes, kpss = det_model.detect(frame, max_num=0, metric='default')
        if len(bboxes) > 0:
            for i in range(len(bboxes)):
                bbox = bboxes[i, :4]
                kps = kpss[i]
                face = Face(bbox=bbox, kps=kps, det_score=bboxes[i, 4])
                rec_model.get(frame, face)  # face embedding from the recognition model
                test_embedding = face.normed_embedding  # extract face embedding
                pred_name, match_score = find_match(test_embedding, known_embeddings, known_names, threshold)  # Find a match

                x1_e, y1_e, x2_e, y2_e = agrandir_bbox_epaules(bbox, frame.shape)

                # color-coding based on recognition
                if pred_name == 'Inconnu':
                    color = (0, 0, 255) # red for unknown
                    label = f"{pred_name}"

                    date_access = datetime.now().strftime("%Y%m%d")
                    time_access = datetime.now().strftime("%H%M%S_")
                    nom_img_ok  = f"inconnu_{date_access + "  " + time_access}.jpg"
                    visage = frame[y1:y2_e, x1_e:x2_e]
                    cv2.imwrite(os.path.join(DOSSIER_INCONNUS, nom_img_ok), visage)
                    enregistrer_acces("Inconnu", "REFUSE", match_score, image=nom_img_ok)
                    
                else:
                    color = (0, 255, 0)  # green for known
                    label = f"{pred_name} ({match_score:.2f})"

                    date_access = datetime.now().strftime("%Y%m%d")
                    time_access = datetime.now().strftime("%H%M%S_")
                    nom_fichier = f"{pred_name}_{date_access + "  " + time_access}.jpg"
                    visage = frame[y1_e:y2_e, x1_e:x2_e]
                    cv2.imwrite(os.path.join(DOSSIER_SUCCESS, nom_fichier), visage)
                    enregistrer_acces(pred_name, "SUCCES", threshold, image=nom_fichier)
                    
                x1, y1, x2, y2 = map(int, bbox)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    
                
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 2)

        cv2.imshow("Face Recognition", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()  
    cv2.destroyAllWindows()


In [10]:
known_embeddings, known_names = load_embeddings()
if known_embeddings is not None and known_names is not None:
    face_recognition(known_embeddings, known_names)

In [11]:
import cv2

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)  # ou cv2.CAP_ANY, ou juste cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erreur : impossible d'ouvrir la caméra")
else:
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("Erreur : frame non capturée")
            break

        cv2.imshow("frame", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()